In [1]:
import openai

openai.__version__

'1.66.3'

In [2]:
import time
import base64
from openai import OpenAI

client = OpenAI()

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
def get_token_stats(response):
    token_stats = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }
    return token_stats

def ask_gpt(user_query, base64_image, model="gpt-4o-mini-2024-07-18"):
    start_time = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    { "type": "text", "text": user_query },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                        },
                    },
                ],
            }
        ],
    )
    latency = time.time() - start_time
    answer = response.choices[0].message.content
    token_stats = get_token_stats(response)
    return answer, token_stats, latency

def ask_gpt_with_structured_output(user_query, base64_image, response_format, model="gpt-4o-mini-2024-07-18"):
    start_time = time.time()
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    { "type": "text", "text": user_query },
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}",},},
                ],
            }
        ],
        response_format=response_format
    )
    latency = time.time() - start_time
    answer = response.choices[0].message.parsed
    token_stats = get_token_stats(response)
    return answer, token_stats, latency

# Object Detection

In [3]:
from pydantic import BaseModel
from typing import List

class Detections(BaseModel):
    detected_class: str
    x_min: float
    x_max: float
    y_min: float
    y_max: float

class Detections(BaseModel):
    detections: List[Detections]

user_query = "Detect the food and its location in the image. " \
"Normalize the coordinates between 0 and 1 based on image width and height."

In [4]:
image_path = "images/brunch.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt_with_structured_output(user_query, 
                                                              base64_image, 
                                                              response_format=Detections)

In [5]:
answer.detections

[Detections(detected_class='Eggs Benedict', x_min=0.15, x_max=0.35, y_min=0.3, y_max=0.6),
 Detections(detected_class='Hash Browns', x_min=0.1, x_max=0.15, y_min=0.25, y_max=0.3),
 Detections(detected_class='Fruit Salad', x_min=0.35, x_max=0.55, y_min=0.5, y_max=0.7),
 Detections(detected_class='Dark Bread', x_min=0.7, x_max=0.9, y_min=0.4, y_max=0.5),
 Detections(detected_class='Drink', x_min=0.6, x_max=0.7, y_min=0.2, y_max=0.4)]

In [6]:
token_stats

{'prompt_tokens': 1273, 'completion_tokens': 182, 'total_tokens': 1455}

In [7]:
latency

7.326383829116821

In [ ]:
from PIL import Image
from PIL import ImageDraw


image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)

width, height = image.size

for i, detection in enumerate(answer.detections):
    x1, x2 = detection.x_min * width, detection.x_max * width
    y1, y2 = detection.y_min * height, detection.y_max * height
    draw.rectangle(((x1, y1), (x2, y2)), outline="blue", width=2)
    draw.text((x1, y1), detection.detected_class, fill="red")

image.show()

In [9]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt_with_structured_output(user_query, 
                                                              base64_image, 
                                                              response_format=Detections)

In [10]:
answer.detections

[Detections(detected_class='bacon', x_min=0.5, x_max=0.8, y_min=0.6, y_max=0.75),
 Detections(detected_class='potatoes', x_min=0.3, x_max=0.5, y_min=0.4, y_max=0.6),
 Detections(detected_class='melon', x_min=0.2, x_max=0.4, y_min=0.4, y_max=0.6)]

In [11]:
token_stats

{'prompt_tokens': 593, 'completion_tokens': 111, 'total_tokens': 704}

In [ ]:
from PIL import Image
from PIL import ImageDraw


image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
width, height = image.size

for i, detection in enumerate(answer.detections):
    x1, x2 = detection.x_min * width, detection.x_max * width
    y1, y2 = detection.y_min * height, detection.y_max * height
    draw.rectangle(((x1, y1), (x2, y2)), outline="blue", width=2)
    draw.text((x1, y1), detection.detected_class)

image.show()

# Visual Question Answering

In [13]:
user_query = """
Detect the foods in the image.
What are macro nutrients of each food?
What are micro nutrients of each food?
What is the calorie count of each food?
What is the serving size of each food?
"""

In [14]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [15]:
print(answer)

Based on the image described, the foods appear to be:

1. **Bacon**
2. **Potatoes**
3. **Cantaloupe (or similar melon)**

### Nutritional Information (per common serving sizes):

#### 1. **Bacon**
- **Serving Size:** 1 slice (about 8 grams)
- **Calories:** Approximately 42
- **Macronutrients:**
  - Protein: 3 grams
  - Fat: 3.3 grams
  - Carbohydrates: 0 grams
- **Micronutrients:**
  - Sodium: High
  - Vitamin B12
  - Selenium

#### 2. **Potatoes**
- **Serving Size:** 1 medium (about 150 grams)
- **Calories:** Approximately 130
- **Macronutrients:**
  - Protein: 3 grams
  - Fat: 0.2 grams
  - Carbohydrates: 30 grams
- **Micronutrients:**
  - Vitamin C
  - Potassium
  - Vitamin B6

#### 3. **Cantaloupe (or similar melon)**
- **Serving Size:** 1 cup (about 150 grams)
- **Calories:** Approximately 53
- **Macronutrients:**
  - Protein: 1.5 grams
  - Fat: 0.2 grams
  - Carbohydrates: 13 grams
- **Micronutrients:**
  - Vitamin A
  - Vitamin C
  - Folate

### Summary
The exact nutritional val

In [16]:
token_stats

{'prompt_tokens': 474, 'completion_tokens': 357, 'total_tokens': 831}

In [17]:
image_path = "images/brunch.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [18]:
print(answer)

I can't analyze the image directly, but I can help you identify common foods based on your description of the meal and provide general information about their macronutrients, micronutrients, calorie counts, and serving sizes. 

### 1. Eggs Benedict (with ham):
- **Macronutrients (per serving, 2 eggs, ham, and sauce):**
  - Protein: ~20g
  - Fat: ~30g
  - Carbohydrates: ~5g

- **Micronutrients:**
  - Vitamins: A, D, B12, riboflavin
  - Minerals: Calcium, phosphorus, potassium

- **Calories:** ~350-450 calories

- **Serving Size:** 1 serving (2 poached eggs, ham, and Hollandaise sauce)

---

### 2. Hash Browns:
- **Macronutrients (per serving, ~1 cup):**
  - Protein: ~2g
  - Fat: ~10g
  - Carbohydrates: ~30g

- **Micronutrients:**
  - Vitamins: C, B6
  - Minerals: Potassium, magnesium

- **Calories:** ~150-200 calories

- **Serving Size:** ~1 cup

---

### 3. Mixed Fruit (e.g., cantaloupe, pineapple, grapefruit):
- **Macronutrients (per serving, ~1 cup mixed):**
  - Protein: ~1g
  - Fat:

In [19]:
user_query = """
I have some photos for which I want you to provide me with an estimate of 
the nutritional content. Specifically, I want you to give me an estimate of energy, protein,
total carbohydrate, total fat, dietary fibre, total sugar, saturated fat, polyunsaturated fat,
monounsaturated fat, calcium, iron, vitamin D, sodium, potassium, folate, folic acid, and
vitamin C in the total meal. Please also provide a list of the foods in the photo with an
estimate of the physical weight of each food in grams. Please provide your best point
estimate and do not provide a range. When estimating the weight, please provide a weight
estimate for each individual ingredient you identify in the meal rather than the combined
weights of multiple ingredients." \
"""

In [20]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [21]:
print(answer)

I can't analyze images directly or provide nutritional content based on images. However, I can help you estimate the nutritional content based on typical values for the items you mentioned. Here’s a general estimate for common ingredients that might be in your meal based on typical serving sizes:

### Estimated Ingredients and Weights

1. **Bacon** (3 slices) - Approx. 75g
2. **Potatoes (cooked)** (about 1 medium potato) - Approx. 150g
3. **Cantaloupe** (about 1 cup, diced) - Approx. 150g

### Nutritional Estimates

#### Bacon (75g)
- Energy: ~300 kcal
- Protein: ~22g
- Total Fat: ~24g
- Saturated Fat: ~9g
- Polyunsaturated Fat: ~2g
- Monounsaturated Fat: ~9g
- Sodium: ~900mg

#### Potatoes (150g)
- Energy: ~130 kcal
- Protein: ~3g
- Total Carbohydrates: ~30g
- Total Sugars: ~2g
- Dietary Fiber: ~3g
- Potassium: ~600mg
- Calcium: ~10mg
- Iron: ~1mg

#### Cantaloupe (150g)
- Energy: ~50 kcal
- Protein: ~1g
- Total Carbohydrates: ~12g
- Total Sugars: ~10g
- Dietary Fiber: ~1g
- Vitamin C

# Document Understanding

In [22]:
user_query = "How much did I pay for the meal? How much tax did I pay? " \
"Which date is that and what is the name of the restaurant?"

image_path = "images/restaurant-bill.jpg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [23]:
print(answer)

You paid a total of $85.88 for the meal. The sales tax you paid was $5.88. The date of the meal was December 27, 2022, and the name of the restaurant is Nomade Westport Restaurant.


In [24]:
user_query = "What is the price of Ravioli di Carne? What are its ingredients?"

image_path = "images/menu-pasta-1.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [25]:
print(answer)

The price of Ravioli di Carne is $23.00. 

Its ingredients include beef-stuffed pasta served in a light Bolognese sauce.


In [26]:
user_query = "What is the price of spaghetti allo scoglio? What are its ingredients?"

image_path = "images/menu-pasta-2.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [27]:
print(answer)

The price of **Spaghetti allo scoglio** is **$28.00**. 

Its ingredients include:
- Chopped garlic
- Mussels
- Shrimp
- Scallops
- Clams
- Cherry tomatoes
